# ToysAD-8K Post-freeze Full-pool Release

**Notebook file:** `11r-toysad8k-full-pool-release.ipynb`

This optional notebook starts only after Notebook 10B is complete. It prepares the 4,107 eligible training scenes for a descriptive sensitivity analysis. The primary and official results remain unchanged, and this stage cannot be used to tune the frozen method or review policy.

The model-facing RGB package and evaluator-only labels, geometry and crosswalk are physically separate. Original development, calibration and internal-lock role labels are preserved only inside the trusted evaluator path. The remaining 2,107 scenes are recorded as `eligible_remainder`.

## Execution contract

- The pinned source files and the completed Notebook 10B evidence are verified before any release work starts.
- The eligible pool must contain 4,107 scenes, 545 asset groups, 39 categories and 82,140 RGB images.
- `incremental` mode prepares a bounded number of new scenes and exports one deterministic recovery bundle.
- The release writes an atomic checkpoint before it creates the recovery bundle.
- A later version restores exactly one matching recovery bundle and rejects changed state.
- `full` mode publishes the model and evaluator resources only after complete coverage.
- ClearML receives aggregate counts, one role chart and run receipts. It receives no scene identifiers, labels, geometry or RGB images.

In [ ]:
from __future__ import annotations

from hashlib import sha256
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image as NotebookImage, display

sys.dont_write_bytecode = True
NOTEBOOK_FILE = "11r-toysad8k-full-pool-release.ipynb"
RUN_MODE = os.getenv("VPI_RUN_MODE", "incremental").strip().lower()
if RUN_MODE not in {"incremental", "full"}:
    raise ValueError("VPI_RUN_MODE must be incremental or full")
MAXIMUM_NEW_SCENES = int(os.getenv("VPI_MAXIMUM_NEW_SCENES", "100" if RUN_MODE == "incremental" else "4107"))
OUTPUT_ROOT = Path(os.getenv("VPI_OUTPUT_ROOT", "/kaggle/working/toysad8k_full_pool_release")).expanduser().resolve()
RECOVERY_ROOT = Path(os.getenv("VPI_RECOVERY_OUTPUT", "/kaggle/working/toysad8k_full_pool_release_recovery")).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RECOVERY_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def kaggle_roots() -> list[Path]:
    root = Path("/kaggle/input")
    return sorted((path for path in root.iterdir() if path.is_dir()), key=lambda path: path.as_posix()) if root.is_dir() else []


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *kaggle_roots()))
    for candidate in candidates:
        for resolved in (candidate, candidate / "code"):
            if (resolved / "scripts/run_full_pool_release.py").is_file():
                return resolved.resolve()
    raise FileNotFoundError("Could not locate the Visual Peer Inspector code snapshot")


def find_named_file(name: str, variable: str) -> Path:
    configured = os.getenv(variable)
    if configured and Path(configured).expanduser().is_file():
        return Path(configured).expanduser().resolve()
    matches = sorted({path.resolve() for root in kaggle_roots() for path in root.rglob(name) if path.is_file()})
    if not matches:
        raise FileNotFoundError(f"Could not locate {name}")
    digests = {sha256(path.read_bytes()).hexdigest() for path in matches}
    if len(digests) != 1:
        raise ValueError(f"Mounted copies of {name} differ")
    return matches[0]


def find_root_with(marker: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend(kaggle_roots())
    matches = []
    for root in candidates:
        if (root / marker).is_file():
            matches.append(root.resolve())
        matches.extend(path.parent.resolve() for path in root.rglob(marker) if path.is_file())
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise ValueError(f"Expected one mounted root containing {marker}, found {len(unique)}")
    return unique[0]


def find_model_input(cohort_id: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else kaggle_roots()
    matches = []
    for root in candidates:
        records = [root / "package_record.json"] if (root / "package_record.json").is_file() else []
        records.extend(path for path in root.rglob("package_record.json") if path.is_file())
        for record in records:
            try:
                value = json.loads(record.read_text(encoding="utf-8"))
            except Exception:
                continue
            if value.get("cohort_id") == cohort_id:
                matches.append(record.parent.resolve())
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise ValueError(f"Expected one model package for {cohort_id}, found {len(unique)}")
    return unique[0]


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source = str(CODE_ROOT / "src")
    existing = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source if not existing else source + os.pathsep + existing
    return environment


def clearml_mode(environment: dict[str, str]) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        access = secrets.get_secret("CLEARML_API_ACCESS_KEY")
        secret = secrets.get_secret("CLEARML_API_SECRET_KEY")
    except Exception:
        access = os.getenv("CLEARML_API_ACCESS_KEY")
        secret = os.getenv("CLEARML_API_SECRET_KEY")
    if not access or not secret:
        return "disabled"
    environment.update({
        "CLEARML_API_ACCESS_KEY": access,
        "CLEARML_API_SECRET_KEY": secret,
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
    })
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-tracking-lock.txt")],
            check=True,
            timeout=180,
            env=environment,
        )
    except Exception as error:
        print(f"ClearML setup was unavailable: {type(error).__name__}")
        return "disabled"
    return "online"

## Locate and verify the trusted inputs

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/post-freeze-sensitivity/11r-toysad8k-full-pool-release.ipynb",
)
SOURCE_ROOT = find_root_with("toysad8k_scene_master.json", "VPI_SOURCE_ROOT")
ADMINISTRATIVE_ROOT = find_root_with("restricted_source_map.jsonl", "VPI_ADMINISTRATIVE_ROOT")

official_candidates = []
for root in kaggle_roots():
    for path in root.rglob("evaluation_complete.json"):
        try:
            value = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if value.get("stage") == "10b" and value.get("status") == "complete":
            official_candidates.append(path.parent.resolve())
if len(set(official_candidates)) != 1:
    raise ValueError("Expected one completed Notebook 10B result")
OFFICIAL_RESULT_ROOT = official_candidates[0]

required_source_files = {
    "master": find_named_file("toysad8k_scene_master.json", "VPI_MASTER_JSON"),
    "catalogue": find_named_file("scene_catalogue.sqlite3", "VPI_CATALOGUE"),
    "readiness": find_named_file("readiness_audit.json", "VPI_READINESS_AUDIT"),
    "archive_index": find_named_file("archive_index.sqlite3", "VPI_ARCHIVE_INDEX"),
    "archive": find_named_file("toysAD8K.zip", "VPI_ARCHIVE_ZIP"),
    "archive_manifest": find_named_file("manifest.json", "VPI_ARCHIVE_MANIFEST"),
    "official_split": find_named_file("ToysAD-8K-split.json", "VPI_OFFICIAL_SPLIT"),
}
display({"Notebook 10B verified input located": True, "source inputs located": len(required_source_files)})

## Restore one previous version when required

In [ ]:
resume_root = os.getenv("VPI_RESUME_ROOT")
recovery_arguments = []
if resume_root:
    source = Path(resume_root).expanduser().resolve(strict=True)
    bundles = sorted(source.rglob("full-pool-release-recovery.zip"))
    manifests = sorted(source.rglob("full-pool-release-recovery-manifest.json"))
    if len(bundles) != 1 or len(manifests) != 1:
        raise ValueError("VPI_RESUME_ROOT must contain exactly one release recovery pair")
    recovery_arguments = ["--recovery-bundle", str(bundles[0]), "--recovery-manifest", str(manifests[0])]
display({"previous recovery attached": bool(recovery_arguments), "maximum new scenes": MAXIMUM_NEW_SCENES})

## Prepare or resume the full-pool release

In [ ]:
environment = code_environment()
tracking_mode = clearml_mode(environment)
command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_full_pool_release.py"),
    "--master-json", str(required_source_files["master"]),
    "--catalogue", str(required_source_files["catalogue"]),
    "--readiness-audit", str(required_source_files["readiness"]),
    "--archive-index", str(required_source_files["archive_index"]),
    "--archive-zip", str(required_source_files["archive"]),
    "--archive-manifest", str(required_source_files["archive_manifest"]),
    "--official-split", str(required_source_files["official_split"]),
    "--administrative-root", str(ADMINISTRATIVE_ROOT),
    "--official-result-root", str(OFFICIAL_RESULT_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--maximum-new-scenes", str(MAXIMUM_NEW_SCENES),
    "--recovery-output-directory", str(RECOVERY_ROOT),
    "--clearml-mode", tracking_mode,
    *recovery_arguments,
]
subprocess.run(command, check=True, env=environment)

## Review aggregate release evidence

In [ ]:
safe_root = OUTPUT_ROOT / "safe"
display(pd.read_csv(safe_root / "primary_role_coverage.csv"))
display(NotebookImage(filename=str(safe_root / "primary_role_coverage.png")))
run_record = json.loads((safe_root / "run_record.json").read_text(encoding="utf-8"))
display({key: run_record[key] for key in (
    "status", "completed_scene_count", "remaining_scene_count",
    "descriptive_sensitivity_only", "no_retuning",
)})

## Output boundary

For an incremental run, publish only the recovery directory as a private Kaggle dataset version and attach exactly that version to the next run. For a completed run, publish `full-pool-model-input`, `full-pool-evaluator-only`, and `publish/post-freeze-release` as three separate private resources. Notebook 11A mounts the model input and release receipt only. Notebook 11B mounts the evaluator resource only after the prediction lock exists.